This notebook:

- cleans Lagos restaurant data
- extract Nigerian behavioral signals
- build Nigerian linguistic style models
- detect soft-life language
- detect emotional expressions
- build culturally grounded recommendation features

Cultural adaptation:

Nigerian vocabulary
soft-life signals
Lagos dining psychology
local conversational style
Nigerian recommendation framing

In [ ]:
# LOAD THE LAGOS RESTAURANT DATASET


import pandas as pd

lagos_df = pd.read_csv(
    "../data/external/clean_lagos_restaurants.csv"
)

lagos_df.head()

In [ ]:
# STEP 2 — INSPECT DATASET STRUCTURE

lagos_df.info()

In [ ]:
persona_df = pd.read_csv(
    "../data/processed/persona_profiles.csv"
)

In [ ]:
persona_df.head()

In [ ]:
# STEP 3 - KEEP IMPORTANT COLUMNS

lagos_df = lagos_df[
    [
        "author_name",
        "review_title",
        "review_text",
        "overall_rating",
        "restaurant_name"
    ]
]

In [ ]:
# STEP 4 - CLEAN TEXT COLUMNS

lagos_df["review_text"] = (
    lagos_df["review_text"]
    .fillna("")
    .astype(str)
)

lagos_df["review_title"] = (
    lagos_df["review_title"]
    .fillna("")
    .astype(str)
)

In [ ]:
# STEP 5 - CREATE FULL REVIEW TEXT

lagos_df["full_review"] = (

    lagos_df["review_title"]
    + " "
    + lagos_df["review_text"]
)

In [ ]:
# STEP 6 - NIGERIAN VOCABULARY SIGNALS

nigerian_expressions = {

    "soft_life": [
        "soft life", "premium enjoyment", "luxury vibes", "chill spot", "soft life", "luxury",
        "premium", "classy", "beautiful ambience", "fine dining", "purr"
    ],

    "social_vibes": [
        "vibes", "turn up", "groove", "lit", "music", "dj"
    ],

    "food_enjoyment": [
        "delicious", "tasty", "sweet", "amazing food", "great meal"
    ],

    "casual_slang": [
        "sha", "abi", "wahala", "dey", "no too bad", "pepper", "gist", "vibes", "omo", "no vex", "no gree for anybody"
    ],

    "complaint_style": [
        "somehow", "not worth it", "too expensive", "wahala", "poor service"
    ],

    "pidgin_staples": [
        "abeg", "na wa o", "wahala", "sef", "nko", "abi", "na", "o",
        "ooh", "sha", "kpa", "kwa", "nawa", "mtchew", "chei", "chai",
        "walahi", "biko", "ndo", "jare", "gan", "sabi", "e choke"
    ],
    
    # Casual greetings & exclamations
    "exclamations": [
        "see finish", "see me see trouble", "oga", "madam", "boss",
        "my brother", "my sister", "my guy", "my dear", "babe",
        "sis", "bro", "chief", "alhaji", "mama", "papa"
    ],

    "social_enjoyment": [
        "hangout", "owambe", "groove", "turn up", "outing", "enjoyment" "it's giving", "vibes", "pepper", "gist", "chill spot"
    ],

    # Expressiveness & communication style (Pidgin, humour, directness)
    "expressiveness": [
        "abeg", "na wa o", "seems", "sef", "nko", "abi", "o", "ooh", "omo",
        "who send you", "na so so", "i no send your papa", "e enter"
        "walahi", "mtchew", "chai", "God willing", "not to praise am too much", "you dey whyne?"
    ],

    # Proverbs & wise sayings (often used to justify an opinion)
    "proverbs": [
        "a child who washes hands can eat with elders",
        "the lizard that jumps from a high tree would break its back",
        "when the music changes, the dance must change",
        "the one who throws a stone in the market forgets that others can throw too",
        "a bird that flies off the earth and lands on a tree is not safe from a stone",
        "he who brings kola brings life",
        "the way you dress is how you will be addressed",
        "it is not the size of the yam that matters, but the size of the stew",
        "a person who is chasing a rat cannot see the antelope",
        "if you want to hide a corpse, put it under a woman's wrapper"
    ],
    
    # Idiomatic expressions (figurative, not literal)
    "idioms": [
        "carry last",        # finish last / be embarrassed
        "chop breakfast",    # suffer a harsh disappointment
        "see finish",        # see someone's true colours / be fed up
        "form 419",          # act fraudulent or fake
        "blow grammar",      # speak overly fancy English
        "show pepper",       # be aggressive or tough
        "catch cruise",      # have fun / joke around
        "give attitude",     # behave rudely or arrogantly
        "carry go",          # take away / steal
        "use your head",     # think properly
        "shine your eye",    # be vigilant, don’t be fooled
        "do the needful",    # take necessary action
        "pull down",         # criticise or undermine someone
        "call somebody",     # confront or challenge
        "run mad",           # malfunction / go crazy
        "enter one chance",  # fall into a trap or irreversible situation
        "hot cake",          # very popular in demand
        "no gree for anybody" # stand your ground, don’t give in
    ],
    
    # Greetings & social expressions (used to open or close reviews)
    "greetings": [
        "how far?", "how now?", "how body?", "how market?",
        "hello o", "good morning o", "good afternoon o", "good evening o",
        "thank you jare", "thanks a lot", "appreciate",
        "sorry o", "my bad", "no wahala"
    ],
    
    # Exclamations & emotional outbursts (strong feelings)
    "exclamations": [
        "chai!", "chei!", "mtchew!", "no way!", "kpa!", "nawa o!",
        "God forbid!", "never!", "ehn?", "bawo?", "see glass!", "alas!!",
        "oga at the top!", "hallelujah!", "e shock you?", "e don happen!"
    ],
    
    # Figurative descriptions (vivid, often exaggerated)
    "figurative_descriptions": [
        "hot like suya",           # very hot
        "sweet like honey",        # delicious
        "bitter like agbo",        # very bitter
        "hard like rock",          # extremely hard/tough
        "soft like cotton",        # very soft
        "smooth like butter",      # very smooth
        "long like express",       # very long
        "slow like snail",         # extremely slow
        "fast like wind",          # very fast
        "small like ant",          # tiny
        "full like church on Sunday"  # very crowded
        "you dey whyne?" #are you joking?
    ],
    
    # Conditional & hypothetical phrases (storytelling markers)
    "conditional_phrases": [
        "if to say", "suppose say", "even if", "whether",
        "unless e be say", "as if", "imagine say", "make e be like say"
    ],
    
    # Persuasion & emphasis (used to convince reader)
    "persuasion": [
        "I swear down", "I swear for you", "believe me",
        "take it from me", "mark my word", "I guarantee you",
        "e no go better for you if you doubt", "try me"
    ],
    
    # Blame & criticism expressions
    "blame_criticism": [
        "the fault na", "na him cause am", "who send you?",
        "you no try", "e no correct", "wrong delivery",
        "na scam", "wayo", "419", "fake", "junk", "trash"
    ],
    
    # Humour & sarcasm markers
    "humour_sarcasm": [
        "I laff", "lolz", "mtchew", "see comedy", "joke of the year",
        "e be like film trick", "movie scene", "story for the gods",
        "you won't believe", "as if I never see", "everywhere first blur"
    ],
    
    # Cultural references (places, brands, events)
    "cultural_references": [
        "computer village",           # tech hub in Lagos
        "Alaba market",               # electronics market
        "Oshodi market",              # busy market
        "Balogun market",             # textile market
        "NEPA", "PHCN",               # electricity company
        "MTN", "Glo", "Airtel",       # network providers
        "BBNaija",                    # reality TV show
        "Nollywood",                  # film industry
        "Asake", "Burna Boy",         # popular musicians
        "Dangote",                    # conglomerate
        "Sallah", "Christmas",        # festivals
        "Ember months"                # September–December
    ],

    "practical_survival": [
        "traffic", "expensive", "affordable", "stress", "queue", "delay"
    ],

    "time_efficiency": [
        "wait time", "delay", "fast delivery", "slow", "African time",
        "hours", "minutes", "late", "early", "prompt", "wasted my time",
        "traffic", "Lagos traffic", "delivered on time"
    ],
    
    # Nigerian‑specific storytelling markers (colloquial narrative style)
    "naija_narrative": [
        "so I tell am", "the guy come say", "I just dey go",
        "immediately I enter", "see as e be", "before I know",
        "the next thing", "as I was coming", "I reach there",
        "the seller tell me", "my sister say make I try am",
        "story for another day", "I no fit shout", "you won't believe"
    ],
    
    # Exaggerated storytelling (typical of oral tradition)
    "hyperbole_narrative": [
        "I waited for years", "the longest hour of my life",
        "everybody in Lagos", "the whole market", "I almost died",
        "I swear down", "e be like film", "like a movie scene"
    ],


    # Kinship & relational terms (used to address or refer)
    "kinship": [
        "oga", "madam", "massa", "boss", "chief", "alhaji",
        "my brother", "my sister", "my guy", "my dear", "my friend",
        "uncle", "aunty", "papa", "mama", "baba", "iya", "bros"
    ],
    
    # Conjunctions & connectors (oral style flow)
    "oral_connectors": [
        "so I tell am", "immediately", "the next thing", "before I know",
        "as I dey go", "come see", "lo and behold", "to cut the long story short",
        "long story short", "in short", "and all that", "and so on"
    ]
}

In [ ]:
# STEP 7 — CREATE SIGNAL DETECTOR

def detect_nigerian_signals(text):

    text = text.lower()

    detected = []

    for category, words in nigerian_expressions.items():

        for word in words:

            if word in text:

                detected.append(category)
                break

    return detected

In [ ]:
# STEP 8 — APPLY SIGNAL DETECTION

lagos_df["nigerian_signals"] = (
    lagos_df["full_review"]
    .apply(detect_nigerian_signals)
)

lagos_df[
    [
        "restaurant_name",
        "nigerian_signals"
    ]
].head(15)

In [ ]:
# STEP 8 — CREATE CULTURAL STYLE DETECTOR
# Now we identify communication patterns.
# For example, if a review contains words like "luxury", "classy", or "premium", we might classify it as "soft life" style. 
# If it contains words like "music", "dj", or "vibes", we might classify it as "social vibes" style. 
# If it contains words like "expensive", "worth it", or "price", we might classify it as "value sensitive" style. Otherwise, we can classify it as "neutral".

def classify_nigerian_style(text):

    text = text.lower()

    if any(
        word in text
        for word in [
            "luxury", "classy", "premium", "ambience", "purr", "aesthetic", "fine dining",
            "rooftop", "chill", "soft life", "VIP", "exclusive", "expensive but worth it",
            "valet", "instagrammable", "date night", "romantic"
        ]
    ):

        return "soft_life"


    elif any(
        word in text
        for word in [
            "music", "dj", "vibes", "groove", "chill"
        ]
    ):

        return "social_vibes"


    elif any(
        word in text
        for word in [
            "expensive", "worth it", "cheap", "price", "value for money"
        ]
    ):
    
        return "social_vibes"


    elif any(
        word in text
        for word in [
            "delay", "fast delivery", "slow", "African time", "traffic", "Lagos traffic", "delivered on time"
        ]
    ):
    
        return "time_sensitive"
    

    elif any(
        word in text
        for word in [
            "buka", "local", "traditional", "amala", "egusi", "eba", "fufu",
            "swallow", "ofada", "native", "home style", "authentic", "village",
            "grandma", "street food", "mama put", "canteen"
        ]
    ):
    
        return "authentic_local"
    

    elif any(
        word in text
        for word in [
            "family", "children", "kids", "everyone", "group", "party",
            "celebration", "birthday", "wedding", "with my people",
            "my friend", "my sister", "my brother", "uncle", "aunty"
        ]
    ):
    
        return "family_oriented"
    

    elif any(
        word in text
        for word in [
           "rude", "attentive", "service", "staff", "waiter", "customer care",
            "friendly", "polite", "ignored", "follow me around", "pushy",
            "respect", "attitude", "shouted", "welcomed", "ignored", "helpful", "unhelpful"
        ]
    ):
    
        return "customer_service"
    

    elif any(
        word in text
        for word in [
            "manage", "sapa", "budget", "cheap", "affordable", "value for money",
            "not worth it", "overpriced", "waste of money", "hustle", "economy",
            "pricey", "my money", "cost", "naira", "discount", "change"
        ]
    ):
    
        return "hustle_minded"
    

    elif any(
        word in text
        for word in [
            "chai", "chei", "mtchew", "nawa o", "worst ever", "omo",
            "run away", "scam", "fake life", "story for the gods",
            "never again", "who send you", "see finish", "carry last"
        ]
    ):
    
        return "sarcastic_dramatic"

    else:

        return "neutral"

In [ ]:
# STEP 9 — APPLY STYLE CLASSIFICATION

lagos_df["cultural_style"] = (
    lagos_df["full_review"]
    .apply(classify_nigerian_style)
)

lagos_df[
    [
        "restaurant_name",
        "cultural_style"
    ]
].head(20)

In [ ]:
# STEP 10 — INSPECT REAL CULTURAL REVIEWS

lagos_df[
    [
        "restaurant_name",
        "overall_rating",
        "full_review",
        "cultural_style"
    ]
].sample(15)

In [ ]:
# STEP 11 — CREATE CULTURAL PROFILE SUMMARY

cultural_summary = (

    lagos_df["cultural_style"]
    .value_counts()
)

cultural_summary

In [ ]:
# STEP 12 — CREATE NIGERIAN PERSONA MAPPING
# Now we connect personas to Nigerian styles.

nigerian_persona_map = {

    "Warm Optimist": {

        "social_vibes": 0.9,
        "authentic_local": 0.7,
        "family_oriented": 0.8,
        "customer_service": 0.5
    },

    "Reactive Reviewer": {

        "customer_service": 0.9,
        "hustle_minded": 0.8,
        "time_sensitive": 0.7,
        "sarcastic_dramatic": 0.4
    },

    "Harsh Critic": {

        "value_sensitive": 0.9,
        "sarcastic_dramatic": 0.8,
        "customer_service": 0.7,
        "hustle_minded": 0.6
    },

    "Emotional Storyteller": {

        "soft_life": 0.9,
        "family_oriented": 0.8,
        "social_vibes": 0.7,
        "customer_service": 0.6
    },

    "Deep Experience Analyst": {

        "soft_life": 0.7,
        "authentic_local": 0.9,
        "time_sensitive": 0.8,
        "value_sensitive": 0.6
    }
}

In [ ]:
persona_df["nigerian_styles"] = (
    persona_df["archetype"]
    .map(nigerian_persona_map)
)

persona_df[
    [
        "archetype",
        "nigerian_styles"
    ]
].head().T